<a href="https://colab.research.google.com/github/ialimustufa/python-crash-course/blob/master/notebooks/06_Python_Data_Workflows_for_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 · Python data workflows for ML

| | |
|---|---|
| Time | about 65 minutes |
| Prerequisites | Notebooks 00–05 |
| Environment | Google Colab, or any Python 3 notebook |
| Thread | Bike-share trips in Indian cities — the same story builds toward the capstone |

## How to use this notebook

1. Click **Open in Colab**, then use **File → Save a copy in Drive**.
2. Run cells **from top to bottom**. If names go missing, use **Runtime → Restart session** and **Run all**.
3. When you see **Predict**, decide the output before you run.
4. Practice cells include a working answer so **Run all** never fails (needed for class and CI). To learn, comment out the answer, rewrite it, then run the check.
5. Hints sit under each practice. Clean solutions are at the end.

Concept sections include diagrams of the idea, not only the code. They show as pictures in Colab and as Mermaid on GitHub.

## What you will be able to do

- Clean a list of dictionaries: types, missing values, duplicates
- Filter, sort, group, and aggregate without pandas
- Encode a categorical column as integer codes or one-hot bits
- Reshape a table as dict-of-lists (pandas) and list-of-lists X (sklearn)
- Separate features from a target column
- Make a deterministic train/validation split and scale with train-only statistics
- Explain why a seeded shuffle exists (`random_state` later)


## Recap warm-up


In [ ]:
row = {"hours": "2", "km": "8.5"}
print(int(row["hours"]), float(row["km"]))


## Why this notebook exists

NumPy, pandas, and scikit-learn all expect **tidy rows**: consistent keys, numeric types, no surprise blanks, and a target you did not accidentally leak into the features.

You will do that work in pure Python. The same ideas map onto `DataFrame.dropna`, `train_test_split`, and `MinMaxScaler` later.

Libraries are objects with methods: `model.fit(X, y)`. A tiny class below shows attributes and methods without becoming an OOP course.

![Tidy rows in Python are the same idea libraries will automate later](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgcmF3WyJyYXcgcm93cyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdIC0tPiBjbGVhblsidHlwZSwgZHJvcCwgZGVkdXBlJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0KICBjbGVhbiAtLT4gdGlkeVsibGlzdCBvZiBkaWN0aW9uYXJpZXMmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQogIHRpZHkgLS0-IGxhdGVyWyJwYW5kYXMgYW5kIHNrbGVhcm4gbGF0ZXImbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQ==)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart TB
  raw[raw rows] --> clean["type, drop, dedupe"]
  clean --> tidy[list of dictionaries]
  tidy --> later[pandas and sklearn later]
```

</details>


In [ ]:
class Table:
    """A tiny stand-in for the object style you will meet in ML libraries."""

    def __init__(self, rows):
        self.rows = rows

    @property
    def n_rows(self):
        return len(self.rows)

    def column(self, name):
        return [row[name] for row in self.rows]

raw_rows = [
    {"id": "1", "city": "Mumbai", "hours": "2", "km": "8.0", "returned": "yes"},
    {"id": "2", "city": "Pune", "hours": "", "km": "3.0", "returned": "yes"},
    {"id": "3", "city": "Mumbai", "hours": "1", "km": "4.0", "returned": "no"},
    {"id": "4", "city": "Goa", "hours": "abc", "km": "5.0", "returned": "yes"},
    {"id": "5", "city": "Pune", "hours": "3", "km": "9.0", "returned": "yes"},
    {"id": "5", "city": "Pune", "hours": "3", "km": "9.0", "returned": "yes"},
    {"id": "6", "city": "Mumbai", "hours": "2", "km": "18.0", "returned": "no"},
]
print("raw rows", Table(raw_rows).n_rows)


## Clean, then transform

![Clean by dropping bad rows before you transform the rest](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgcmF3Um93WyJlYWNoIHJhdyByb3cmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXSAtLT4gZHVweyJkdXBsaWNhdGUgaWQ_Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7In0KICBkdXAgLS0-fHllc3wgZHJvcER1cFsiZHJvcCBkdXBsaWNhdGUmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQogIGR1cCAtLT58bm98IHBhcnNlWyJwYXJzZSBob3VycyBhbmQga20mbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQogIHBhcnNlIC0tPiBiYWR7Im1pc3Npbmcgb3IgaW52YWxpZD8mbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsifQogIGJhZCAtLT58eWVzfCBkcm9wQmFkWyJkcm9wIHRoZSByb3cmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQogIGJhZCAtLT58bm98IGtlZXBbImtlZXAgYSB0eXBlZCByb3cmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQ==)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart TB
  rawRow[each raw row] --> dup{"duplicate id?"}
  dup -->|yes| dropDup[drop duplicate]
  dup -->|no| parse[parse hours and km]
  parse --> bad{"missing or invalid?"}
  bad -->|yes| dropBad[drop the row]
  bad -->|no| keep[keep a typed row]
```

</details>


In [ ]:
def parse_float(text):
    try:
        return float(text)
    except (TypeError, ValueError):
        return None


def clean_rows(rows):
    cleaned = []
    seen_ids = set()
    report = {"dropped_missing": 0, "dropped_duplicate": 0, "dropped_invalid": 0}
    for row in rows:
        if row["id"] in seen_ids:
            report["dropped_duplicate"] += 1
            continue
        hours = parse_float(row["hours"])
        km = parse_float(row["km"])
        if hours is None or km is None:
            if row["hours"] in (None, "") or row["km"] in (None, ""):
                report["dropped_missing"] += 1
            else:
                report["dropped_invalid"] += 1
            continue
        seen_ids.add(row["id"])
        cleaned.append(
            {
                "id": int(row["id"]),
                "city": row["city"],
                "hours": hours,
                "km": km,
                "returned": row["returned"] == "yes",
                "km_per_hour": km / hours,
            }
        )
    return cleaned, report

trips, quality = clean_rows(raw_rows)
print(trips)
print(quality)


In [ ]:
mumbai = [row for row in trips if row["city"] == "Mumbai"]
by_km = sorted(trips, key=lambda row: row["km"], reverse=True)

totals = {}
for row in trips:
    totals[row["city"]] = totals.get(row["city"], 0) + row["km"]
print("Mumbai rows", len(mumbai))
print("farthest id", by_km[0]["id"])
print("km by city", totals)


## Features, target, split, scale

The **target** is the thing you would predict (`returned`). **Features** are the inputs.

Split **before** you compute min/max or averages. If the validation rows influence scaling, you have leaked information.

This split is deterministic: sort by `id`, first 80% train, remainder validation. No random shuffle, so CI stays stable.

![Split first, then fit scaling on train only](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgcm93c1sidGlkeSByb3dzJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0gLS0-IHh5WyJmZWF0dXJlcyBYIGFuZCB0YXJnZXQgeSZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgeHkgLS0-IHNwbGl0Rmlyc3RbInNwbGl0IGJ5IGlkJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0KICBzcGxpdEZpcnN0IC0tPiB0cmFpblsidHJhaW4mbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQogIHNwbGl0Rmlyc3QgLS0-IHZhbGlkWyJ2YWxpZCZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgdHJhaW4gLS0-IGZpdFsiZml0IG1pbi1tYXggb24gdHJhaW4mbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQogIGZpdCAtLT4gc2NhbGVUWyJzY2FsZSB0cmFpbiZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgZml0IC0tPiBzY2FsZVZbInNjYWxlIHZhbGlkJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0=)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart TB
  rows[tidy rows] --> xy[features X and target y]
  xy --> splitFirst[split by id]
  splitFirst --> train[train]
  splitFirst --> valid[valid]
  train --> fit[fit min-max on train]
  fit --> scaleT[scale train]
  fit --> scaleV[scale valid]
```

</details>

![Scaling before the split lets validation rows influence train](https://mermaid.ink/svg/Zmxvd2NoYXJ0IExSCiAgd3JvbmdbInNjYWxlIGFsbCByb3dzIGZpcnN0Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0gLS0-IGxlYWtbInZhbGlkYXRpb24gbGVha2VkJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0KICByaWdodFsic3BsaXQgZmlyc3QmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXSAtLT4gbm9MZWFrWyJ0cmFpbi1vbmx5IGJvdW5kcyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJd)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart LR
  wrong[scale all rows first] --> leak[validation leaked]
  right[split first] --> noLeak[train-only bounds]
```

</details>


In [ ]:
def split_rows(rows, fraction=0.8):
    ordered = sorted(rows, key=lambda row: row["id"])
    cut = int(len(ordered) * fraction)
    if cut == 0 or cut == len(ordered):
        raise ValueError("split would leave an empty side")
    return ordered[:cut], ordered[cut:]


def min_max_fit(rows, columns):
    stats = {}
    for column in columns:
        values = [row[column] for row in rows]
        stats[column] = {"min": min(values), "max": max(values)}
    return stats


def min_max_transform(rows, stats):
    scaled = []
    for row in rows:
        item = dict(row)
        for column, bounds in stats.items():
            span = bounds["max"] - bounds["min"]
            item[column] = 0.0 if span == 0 else (row[column] - bounds["min"]) / span
        scaled.append(item)
    return scaled

feature_cols = ["hours", "km"]
train, valid = split_rows(trips)
stats = min_max_fit(train, feature_cols)
train_scaled = min_max_transform(train, stats)
valid_scaled = min_max_transform(valid, stats)
print("train ids", [row["id"] for row in train])
print("valid ids", [row["id"] for row in valid])
print("train-only scaler", stats)

leaky_max = max(row["km"] for row in trips)
train_max = stats["km"]["max"]
print("km max on all rows", leaky_max)
print("km max on train only", train_max)
print("leaking would shrink train values because the 18 km row sits in validation")


## Categories and reproducibility

Models need numbers. A city name becomes an integer **code**. Sort the unique cities so the mapping does not depend on row order. Fit the mapping on **train** in real work (same rule as scaling). Here the table is tiny, so we show the idea on all cleaned rows.

scikit-learn's `train_test_split(..., random_state=42)` is a **seeded shuffle**. This course sorts by `id` instead, so CI stays stable. If you shuffle, copy the list first and seed the generator.

![Sorted unique labels become integer codes](https://mermaid.ink/svg/Zmxvd2NoYXJ0IExSCiAgY2l0aWVzWyJ1bmlxdWUgY2l0aWVzJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0gLS0-IHNvcnRlZENbInNvcnQmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQogIHNvcnRlZEMgLS0-IGNvZGVzWyJNdW1iYWkgMCwgUHVuZSAxJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0=)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart LR
  cities[unique cities] --> sortedC[sort]
  sortedC --> codes["Mumbai 0, Pune 1"]
```

</details>


In [ ]:
city_levels = sorted({row["city"] for row in trips})
city_code = {city: index for index, city in enumerate(city_levels)}
encoded = []
for row in trips:
    item = dict(row)
    item["city_code"] = city_code[row["city"]]
    encoded.append(item)
print("city_code", city_code)
print([row["city_code"] for row in encoded])

import random

ids = [row["id"] for row in trips]
shuffled = ids[:]
random.Random(0).shuffle(shuffled)
print("seeded shuffle", shuffled)
print("id split stays", [row["id"] for row in train], [row["id"] for row in valid])


## Three shapes of the same table

The same trips appear three ways. pandas `DataFrame(by_column)` wants a **dict of lists** (one list per column). sklearn `model.fit(X, y)` wants `X` as a **list of lists of numbers**. A **list of dicts** is the shape you cleaned.

A city can also become **one-hot** bits: Mumbai → `[1, 0]`, Pune → `[0, 1]`. An unseen city (Goa) becomes `[0, 0]` — sklearn `handle_unknown="ignore"`. Fit the level list on **train** only.

![One table, three shapes: rows, columns, numeric matrix](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgcm93c1sibGlzdCBvZiBkaWN0cyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdIC0tPiBjb2xzWyJkaWN0IG9mIGxpc3RzJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0KICByb3dzIC0tPiBtYXRyaXhbImxpc3Qgb2YgbGlzdHMgWCZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJd)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart TB
  rows[list of dicts] --> cols["dict of lists"]
  rows --> matrix["list of lists X"]
```

</details>


In [ ]:
by_column = {
    "hours": [row["hours"] for row in trips],
    "km": [row["km"] for row in trips],
}
X_numeric = [[row["hours"], row["km"]] for row in trips]
y_target = [row["returned"] for row in trips]
print("column hours", by_column["hours"])
print("X rows", len(X_numeric), "y rows", len(y_target))

train_levels = sorted({row["city"] for row in train})

def one_hot(city, levels):
    return [int(city == level) for level in levels]

print("levels", train_levels)
print("Mumbai", one_hot("Mumbai", train_levels))
print("unseen Goa", one_hot("Goa", train_levels))


## Practice 06.1


Return only rows whose `hours` is at least 2.


In [ ]:
# Practice 06.1
long_enough = [row for row in trips if row["hours"] >= 2]
print(long_enough)

# Check 06.1
assert [row["id"] for row in long_enough] == [1, 5, 6]
print("Practice 06.1 passed.")


<details>
<summary>Hint 06.1</summary>

A list comprehension with `if row["hours"] >= 2`.

</details>


## Practice 06.2


Compute mean `km` grouped by `city`.


In [ ]:
# Practice 06.2
sums = {}
counts = {}
for row in trips:
    city = row["city"]
    sums[city] = sums.get(city, 0.0) + row["km"]
    counts[city] = counts.get(city, 0) + 1
mean_km = {city: sums[city] / counts[city] for city in sums}
print(mean_km)

# Check 06.2
assert abs(mean_km["Mumbai"] - 10.0) < 1e-9
print("Practice 06.2 passed.")


<details>
<summary>Hint 06.2</summary>

Keep a running sum and a count per city, then divide.

</details>


## Practice 06.3


Build `X` (list of `[hours, km]`) and `y` (list of `returned`) from `trips`.


In [ ]:
# Practice 06.3
X = [[row["hours"], row["km"]] for row in trips]
y = [row["returned"] for row in trips]
print(X, y)

# Check 06.3
assert X[0] == [2.0, 8.0]
assert y == [True, False, True, False]
print("Practice 06.3 passed.")


<details>
<summary>Hint 06.3</summary>

Two comprehensions: one nested list for features, one list for the boolean target.

</details>


## Practice 06.4


Map each city in `trips` to an integer code using sorted unique names. Store the mapping in `city_code`.


In [ ]:
# Practice 06.4
city_levels = sorted({row["city"] for row in trips})
city_code = {city: index for index, city in enumerate(city_levels)}
print(city_code)

# Check 06.4
assert city_code["Mumbai"] == 0
assert city_code["Pune"] == 1
assert set(city_code) == {"Mumbai", "Pune"}
print("Practice 06.4 passed.")


<details>
<summary>Hint 06.4</summary>

Unique cities via `{row["city"] for row in trips}`, then `sorted`, then `enumerate`.

</details>


## Practice 06.5


Build `by_column` as a dict of `hours` and `km` lists from `trips`. Then one-hot the unseen city `"Goa"` against `train_levels = ["Mumbai", "Pune"]`.


In [ ]:
# Practice 06.5
by_column = {
    "hours": [row["hours"] for row in trips],
    "km": [row["km"] for row in trips],
}
train_levels = ["Mumbai", "Pune"]
goa_one_hot = [int("Goa" == level) for level in train_levels]
print(by_column)
print(goa_one_hot)

# Check 06.5
assert by_column["hours"][0] == 2.0
assert len(by_column["hours"]) == len(trips) == len(by_column["km"])
assert goa_one_hot == [0, 0]
print("Practice 06.5 passed.")


<details>
<summary>Hint 06.5</summary>

Two comprehensions fill the dict. `[int("Goa" == level) for level in train_levels]` is all zeros.

</details>


## Common mistakes

- Scaling the full dataset, then splitting. Fit on train only.
- Leaving the target inside `X`.
- Using `set` iteration order when you need a stable split or stable category codes — sort first.
- Shuffling in place without a copy (`random.shuffle(ids)` mutates `ids`).
- Passing a list of dicts to `model.fit`. sklearn wants a numeric `X`.
- Fitting category codes on train+valid, then wondering why validation cities leaked into the map.


## Recap


Cleaning, splitting, scaling, and encoding are Python problems first. Libraries automate the same steps; they do not replace them.


## Check your understanding

<details>
<summary>When should you compute min/max for scaling?</summary>

On the training rows only, then apply those bounds to validation.

</details>

<details>
<summary>Where does the target column belong?</summary>

Out of X, stored separately as y. If it stays in X, a model can cheat.

</details>

<details>
<summary>Why sort city names before assigning codes?</summary>

So the mapping is stable. Unordered set iteration can shuffle codes if row order changes.

</details>

<details>
<summary>What shape does sklearn model.fit(X, y) expect for X?</summary>

A list of lists of numbers (rows = samples). A dict of lists is the pandas constructor shape.

</details>


## Solutions


<details>
<summary>Solution 06.1</summary>

```python
long_enough = [row for row in trips if row["hours"] >= 2]
```

</details>


<details>
<summary>Solution 06.2</summary>

```python
sums, counts = {}, {}
for row in trips:
    sums[row['city']] = sums.get(row['city'], 0.0) + row['km']
    counts[row['city']] = counts.get(row['city'], 0) + 1
mean_km = {city: sums[city] / counts[city] for city in sums}
```

</details>


<details>
<summary>Solution 06.3</summary>

```python
X = [[row["hours"], row["km"]] for row in trips]
y = [row["returned"] for row in trips]
```

</details>


<details>
<summary>Solution 06.4</summary>

```python
city_levels = sorted({row["city"] for row in trips})
city_code = {city: index for index, city in enumerate(city_levels)}
```

</details>


<details>
<summary>Solution 06.5</summary>

```python
by_column = {
    "hours": [row["hours"] for row in trips],
    "km": [row["km"] for row in trips],
}
goa_one_hot = [int("Goa" == level) for level in train_levels]
```

</details>


## Navigation

- Previous: [05_Errors_Modules_and_Files.ipynb](https://github.com/ialimustufa/python-crash-course/blob/master/notebooks/05_Errors_Modules_and_Files.ipynb) · [Open in Colab](https://colab.research.google.com/github/ialimustufa/python-crash-course/blob/master/notebooks/05_Errors_Modules_and_Files.ipynb)
- Next: [07_Fundamentals_Capstone.ipynb](https://github.com/ialimustufa/python-crash-course/blob/master/notebooks/07_Fundamentals_Capstone.ipynb) · [Open in Colab](https://colab.research.google.com/github/ialimustufa/python-crash-course/blob/master/notebooks/07_Fundamentals_Capstone.ipynb)
- Course home: [README.md](https://github.com/ialimustufa/python-crash-course/blob/master/README.md)
